# SPARC Articulatory Feature Extraction - Colab

This notebook encodes LibriSpeech audio into articulatory features using SPARC.
Run on Colab with a T4/V100 GPU for 5-10x speedup over Mac M4.

**Steps:**
1. Install dependencies
2. Download LibriSpeech directly (fast on Google servers)
3. Run SPARC encoding on GPU
4. Save to Google Drive for download

**Runtime:** Change runtime to GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
# Step 0: Verify GPU and mount Google Drive for saving progress
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU')

# Mount Google Drive - features will be saved directly here so nothing is lost
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_FEATURES = '/content/drive/MyDrive/articulatory_tts/features'
os.makedirs(DRIVE_FEATURES, exist_ok=True)
print(f'Google Drive mounted. Features will save to: {DRIVE_FEATURES}')
existing = len([f for f in os.listdir(DRIVE_FEATURES) if f.endswith('.npz') and f != 'norm_stats.npz'])
print(f'Already encoded on Drive: {existing} files')

In [ ]:
# Step 1: Install SPARC
!pip install -q speech-articulatory-coding soundfile

In [ ]:
# Step 2: Download LibriSpeech train-clean-100 and dev-clean
import os

if not os.path.exists('LibriSpeech/train-clean-100'):
    print('Downloading train-clean-100 (6.3GB)...')
    !wget -q --show-progress http://www.openslr.org/resources/12/train-clean-100.tar.gz
    !tar xzf train-clean-100.tar.gz
    !rm train-clean-100.tar.gz
    print('Done!')
else:
    print('train-clean-100 already downloaded.')

if not os.path.exists('LibriSpeech/dev-clean'):
    print('Downloading dev-clean (337MB)...')
    !wget -q --show-progress http://www.openslr.org/resources/12/dev-clean.tar.gz
    !tar xzf dev-clean.tar.gz
    !rm dev-clean.tar.gz
    print('Done!')
else:
    print('dev-clean already downloaded.')

import glob
train_files = sorted(glob.glob('LibriSpeech/train-clean-100/**/*.flac', recursive=True))
dev_files = sorted(glob.glob('LibriSpeech/dev-clean/**/*.flac', recursive=True))
print(f'train-clean-100: {len(train_files)} files')
print(f'dev-clean: {len(dev_files)} files')
print(f'Total: {len(train_files) + len(dev_files)} files')

In [ ]:
# Step 3: Load SPARC model on GPU
from sparc import load_model
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading SPARC on {device}...')
coder = load_model('en', device=device)
print('SPARC loaded!')

# Speed test on 3 files to get stable estimate
times = []
for f in (dev_files + train_files)[:3]:
    t0 = time.time()
    code = coder.encode(f)
    times.append(time.time() - t0)

avg_time = sum(times) / len(times)
total_files = len(train_files) + len(dev_files)
print(f'Speed: {avg_time:.2f}s per file')
print(f'Estimated total time: {avg_time * total_files / 3600:.1f} hours for {total_files} files')

In [ ]:
# Step 4: Encode ALL files
# Saves to local disk (fast), syncs to Google Drive every 100 files (safe)
import numpy as np
import json
import glob
import shutil
from pathlib import Path
from tqdm.notebook import tqdm

DRIVE_FEATURES = '/content/drive/MyDrive/articulatory_tts/features'
LOCAL_FEATURES = '/content/features'
os.makedirs(LOCAL_FEATURES, exist_ok=True)
os.makedirs(DRIVE_FEATURES, exist_ok=True)

# Re-find audio files (in case session restarted)
train_files = sorted(glob.glob('LibriSpeech/train-clean-100/**/*.flac', recursive=True))
dev_files = sorted(glob.glob('LibriSpeech/dev-clean/**/*.flac', recursive=True))
all_files = dev_files + train_files
print(f'Total audio files: {len(all_files)}')

# Check what's already done on Drive
drive_done = {p.stem for p in Path(DRIVE_FEATURES).glob('*.npz') if p.stem != 'norm_stats'}
print(f'Already on Drive: {len(drive_done)} files')

speaker_embs = {}
success, failed, skipped = 0, 0, 0
local_unsaved = 0
t_start = time.time()

for f in tqdm(all_files, desc='Encoding'):
    utt_id = Path(f).stem
    spk_id = utt_id.split('-')[0]

    # Skip if already on Drive
    if utt_id in drive_done:
        skipped += 1
        continue

    # Skip if already on local disk
    local_path = Path(LOCAL_FEATURES) / f'{utt_id}.npz'
    if local_path.exists():
        skipped += 1
        local_unsaved += 1
    else:
        try:
            code = coder.encode(f)
            ema = np.asarray(code['ema'], dtype=np.float32)
            pitch = np.asarray(code['pitch'], dtype=np.float32).squeeze(-1)
            loudness = np.asarray(code['loudness'], dtype=np.float32).squeeze(-1)
            spk_emb = np.asarray(code['spk_emb'], dtype=np.float32)

            min_len = min(ema.shape[0], pitch.shape[0], loudness.shape[0])
            ema, pitch, loudness = ema[:min_len], pitch[:min_len], loudness[:min_len]

            np.savez_compressed(str(local_path), ema=ema, pitch=pitch, loudness=loudness, spk_emb=spk_emb)

            if spk_id not in speaker_embs:
                speaker_embs[spk_id] = []
            speaker_embs[spk_id].append(spk_emb)
            success += 1
            local_unsaved += 1
        except Exception as e:
            if failed < 5:
                print(f'FAILED {utt_id}: {e}')
            failed += 1

    # Sync to Drive every 100 new files
    if local_unsaved >= 100:
        sync_start = time.time()
        for lf in Path(LOCAL_FEATURES).glob('*.npz'):
            drive_path = Path(DRIVE_FEATURES) / lf.name
            if not drive_path.exists():
                shutil.copy2(str(lf), str(drive_path))
        sync_time = time.time() - sync_start
        local_unsaved = 0

    # Progress update every 500 files
    if (success + skipped) % 500 == 0 and success > 0:
        elapsed = time.time() - t_start
        rate = success / max(elapsed, 1)
        remaining_files = len(all_files) - success - skipped - failed
        remaining_time = remaining_files / max(rate, 0.01)
        total_done = len(drive_done) + success
        print(f'  [{total_done}/{len(all_files)}] {success} new, {skipped} skipped, ~{remaining_time/60:.0f}min left')

# Final sync to Drive
if local_unsaved > 0:
    print(f'\nFinal sync: {local_unsaved} files to Google Drive...')
    for lf in Path(LOCAL_FEATURES).glob('*.npz'):
        drive_path = Path(DRIVE_FEATURES) / lf.name
        if not drive_path.exists():
            shutil.copy2(str(lf), str(drive_path))

elapsed = time.time() - t_start
total_on_drive = len([p for p in Path(DRIVE_FEATURES).glob('*.npz') if p.stem != 'norm_stats'])
print(f'\nDone in {elapsed/3600:.1f} hours: {success} new, {skipped} skipped, {failed} failed')
print(f'Total files on Drive: {total_on_drive}')
print(f'Safe on Google Drive — nothing lost if Colab disconnects!')

In [ ]:
# Step 5: Save speaker embeddings and normalization stats to Drive
import numpy as np
import json
from pathlib import Path

DRIVE_FEATURES = '/content/drive/MyDrive/articulatory_tts/features'
output_dir = Path(DRIVE_FEATURES)

# Rebuild speaker embeddings from all saved files
print('Building speaker embeddings from all saved files...')
speaker_embs = {}
for p in sorted(output_dir.glob('*.npz')):
    if p.stem == 'norm_stats':
        continue
    spk_id = p.stem.split('-')[0]
    try:
        d = np.load(str(p))
        if spk_id not in speaker_embs:
            speaker_embs[spk_id] = []
        speaker_embs[spk_id].append(d['spk_emb'])
    except:
        pass

avg_spk = {k: np.mean(v, axis=0).tolist() for k, v in speaker_embs.items()}
with open(str(output_dir / 'speaker_embeddings.json'), 'w') as f:
    json.dump(avg_spk, f)
print(f'Speaker embeddings: {len(avg_spk)} speakers')

# Compute normalization stats (streaming to avoid memory issues)
print('Computing normalization stats...')
running_sum = np.zeros(14, dtype=np.float64)
running_sq_sum = np.zeros(14, dtype=np.float64)
total_frames = 0

for p in sorted(output_dir.glob('*.npz')):
    if p.stem == 'norm_stats':
        continue
    try:
        d = np.load(str(p))
        pitch_col = d['pitch'][:, None] if d['pitch'].ndim == 1 else d['pitch']
        loud_col = d['loudness'][:, None] if d['loudness'].ndim == 1 else d['loudness']
        feat = np.concatenate([d['ema'], pitch_col, loud_col], axis=-1)
        running_sum += feat.sum(axis=0)
        running_sq_sum += (feat ** 2).sum(axis=0)
        total_frames += feat.shape[0]
    except:
        pass

mean = (running_sum / total_frames).astype(np.float32)
std = np.sqrt(running_sq_sum / total_frames - mean ** 2).astype(np.float32)
std[std < 1e-6] = 1.0

np.savez(str(output_dir / 'norm_stats.npz'), mean=mean, std=std)

n_files = len([p for p in output_dir.glob('*.npz') if p.stem != 'norm_stats'])
print(f'Total files: {n_files}')
print(f'Total frames: {total_frames:,} ({total_frames/50/3600:.1f} hours at 50Hz)')
print(f'All saved to Google Drive: {DRIVE_FEATURES}')

In [ ]:
# Step 6: Download from Google Drive to your Mac
# 
# Option A: Download the folder directly from drive.google.com
#   1. Go to drive.google.com
#   2. Navigate to: articulatory_tts/features
#   3. Right-click the folder → Download (it auto-zips)
#
# Option B: Use gdown on your Mac terminal:
#   pip install gdown
#   gdown --folder "https://drive.google.com/drive/folders/YOUR_FOLDER_ID"
#
# After downloading, extract to:
#   ~/projects/articulatory-tts/data/features_all/

print("Features are saved in Google Drive at: articulatory_tts/features/")
print(f"Total files: {len(list(Path(DRIVE_FEATURES).glob('*.npz')))}")
print()
print("To use on your Mac:")
print("1. Download the 'features' folder from Google Drive")
print("2. Place files in: ~/projects/articulatory-tts/data/features_all/")
print("3. Then run the training commands in the last cell")